# Name: Baraka Mandi

## Coding Problem 1 — Masked Self-Attention

Scaled dot-product attention:

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)

TOKENS = ["Token 1", "Token 2", "Token 3", "Token 4"]

def show(matrix, title):
    # Print a 4x4 matrix with query (rows) and key (columns) labels.
    df = pd.DataFrame(matrix, index=[f"Q: {t}" for t in TOKENS],
                      columns=[f"K: {t}" for t in TOKENS])
    print(f"--- {title} ---")
    print(df.to_string())
    print()

def softmax(x, axis=-1):
    # Numerically stable softmax that also handles -inf entries.
    # If an entire row is -inf (every key masked), that row returns all zeros
    # instead of NaN.
    row_max = np.max(x, axis=axis, keepdims=True)
    row_max = np.where(np.isfinite(row_max), row_max, 0.0)
    exp_x = np.exp(x - row_max)                      # exp(-inf) = 0
    denom = np.sum(exp_x, axis=axis, keepdims=True)
    return np.divide(exp_x, denom, out=np.zeros_like(exp_x), where=denom > 0)

### (a) Scaled Dot-Product Attention

In [2]:
def scaled_dot_product_attention(Q, K, V):
    d_k = K.shape[-1]
    # 1. Attention scores via matrix multiplication: (..., Lq, Lk)
    scores = Q @ np.swapaxes(K, -1, -2)
    # 2. Scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    # 3. Softmax over the key dimension -> attention weights
    weights = softmax(scores, axis=-1)
    # 4. Weighted sum of value vectors -> (..., Lq, d_v)
    output = weights @ V
    return weights, output

In [3]:
# Test data: one sequence of 4 tokens, d_k = d_v = 8
rng = np.random.default_rng(42)
seq_len, d_k, d_v = 4, 8, 8
Q = rng.standard_normal((seq_len, d_k))
K = rng.standard_normal((seq_len, d_k))
V = rng.standard_normal((seq_len, d_v))

weights_a, output_a = scaled_dot_product_attention(Q, K, V)
show(weights_a, "(a) Attention weights, no mask")
print("Output shape:", output_a.shape)

# Sanity checks: each row is a probability distribution, and the result
# matches a direct loop over the formula.
assert np.allclose(weights_a.sum(axis=-1), 1.0)
ref = np.zeros_like(output_a)
for i in range(seq_len):
    s = np.array([Q[i] @ K[j] for j in range(seq_len)]) / np.sqrt(d_k)
    w = np.exp(s - s.max()); w /= w.sum()
    ref[i] = sum(w[j] * V[j] for j in range(seq_len))
assert np.allclose(output_a, ref)
print("Part (a) checks passed.")

--- (a) Attention weights, no mask ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1      0.2941      0.0641      0.4384      0.2035
Q: Token 2      0.2989      0.2388      0.3353      0.1271
Q: Token 3      0.3099      0.2558      0.2948      0.1395
Q: Token 4      0.1928      0.4397      0.1781      0.1893

Output shape: (4, 8)
Part (a) checks passed.


### (b) Causal Masking

In [4]:
def make_causal_mask(seq_len):
    # 1 = allowed, 0 = masked (future position)
    return np.tril(np.ones((seq_len, seq_len), dtype=int))

def to_additive_mask(allowed):
    # Convert a 1/0 'allowed' mask into the additive mask M (0 / -inf).
    return np.where(allowed.astype(bool), 0.0, -np.inf)

causal = make_causal_mask(seq_len)
show(causal, "Causal mask (1 = allowed, 0 = masked)")
show(to_additive_mask(causal), "Additive causal mask M")

--- Causal mask (1 = allowed, 0 = masked) ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1           1           0           0           0
Q: Token 2           1           1           0           0
Q: Token 3           1           1           1           0
Q: Token 4           1           1           1           1

--- Additive causal mask M ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1         0.0        -inf        -inf        -inf
Q: Token 2         0.0         0.0        -inf        -inf
Q: Token 3         0.0         0.0         0.0        -inf
Q: Token 4         0.0         0.0         0.0         0.0



In [5]:
def scaled_dot_product_attention(Q, K, V, causal_mask=False):
    d_k = K.shape[-1]
    Lq, Lk = Q.shape[-2], K.shape[-2]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)

    if causal_mask:
        allowed = np.tril(np.ones((Lq, Lk), dtype=bool))
        scores = scores + to_additive_mask(allowed)   # applied BEFORE softmax

    weights = softmax(scores, axis=-1)
    output = weights @ V
    return weights, output

weights_b, output_b = scaled_dot_product_attention(Q, K, V, causal_mask=True)
show(weights_a, "(b) Attention weights BEFORE masking")
show(weights_b, "(b) Attention weights AFTER causal masking")

assert np.allclose(weights_b[np.triu_indices(seq_len, k=1)], 0.0)  # no future attention
assert np.allclose(weights_b.sum(axis=-1), 1.0)
assert np.allclose(weights_b[0], [1, 0, 0, 0])  # token 1 can only see itself
print("Part (b) checks passed.")

--- (b) Attention weights BEFORE masking ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1      0.2941      0.0641      0.4384      0.2035
Q: Token 2      0.2989      0.2388      0.3353      0.1271
Q: Token 3      0.3099      0.2558      0.2948      0.1395
Q: Token 4      0.1928      0.4397      0.1781      0.1893

--- (b) Attention weights AFTER causal masking ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1      1.0000      0.0000      0.0000      0.0000
Q: Token 2      0.5558      0.4442      0.0000      0.0000
Q: Token 3      0.3602      0.2972      0.3426      0.0000
Q: Token 4      0.1928      0.4397      0.1781      0.1893

Part (b) checks passed.


### (c) Padding Masking

In [6]:
def scaled_dot_product_attention(Q, K, V, causal_mask=False, padding_mask=None):
    # Scaled dot-product attention with optional causal and padding masks.
    #
    # Args:
    #     Q: (..., Lq, d_k) queries
    #     K: (..., Lk, d_k) keys
    #     V: (..., Lk, d_v) values
    #     causal_mask: if True, query i cannot attend to keys j > i
    #     padding_mask: (..., L) array, 1 = valid token, 0 = padding.
    #                   Used on keys (blocks padded columns) and, for
    #                   self-attention, on queries (zeroes padded output rows).
    # Returns:
    #     weights: (..., Lq, Lk) attention weights
    #     output:  (..., Lq, d_v) context vectors
    d_k = K.shape[-1]
    Lq, Lk = Q.shape[-2], K.shape[-2]

    # 1-2. Scaled scores
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)

    # Build the combined boolean 'allowed' mask
    allowed = np.ones(scores.shape, dtype=bool)
    if causal_mask:
        allowed &= np.tril(np.ones((Lq, Lk), dtype=bool))
    if padding_mask is not None:
        pad = np.asarray(padding_mask).astype(bool)
        allowed &= pad[..., np.newaxis, :]            # block padded KEY columns

    # Add M (0 / -inf) BEFORE softmax
    scores = scores + to_additive_mask(allowed)

    # 3. Softmax
    weights = softmax(scores, axis=-1)

    # 4. Weighted sum of values
    output = weights @ V

    # Zero the context output of padded QUERY positions
    if padding_mask is not None:
        output = np.where(pad[..., :, np.newaxis], output, 0.0)

    return weights, output

##### Tests with `padding_mask = [1, 1, 1, 0]`

In [7]:
padding_mask = np.array([1, 1, 1, 0])

w_none, out_none = scaled_dot_product_attention(Q, K, V)
w_causal, out_causal = scaled_dot_product_attention(Q, K, V, causal_mask=True)
w_pad, out_pad = scaled_dot_product_attention(Q, K, V, padding_mask=padding_mask)
w_both, out_both = scaled_dot_product_attention(Q, K, V, causal_mask=True,
                                                padding_mask=padding_mask)

show(w_none,   "BEFORE masking (no mask)")
show(w_causal, "AFTER causal masking")
show(w_pad,    "AFTER padding masking  (padding_mask = [1, 1, 1, 0])")
show(w_both,   "AFTER causal + padding masking")

--- BEFORE masking (no mask) ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1      0.2941      0.0641      0.4384      0.2035
Q: Token 2      0.2989      0.2388      0.3353      0.1271
Q: Token 3      0.3099      0.2558      0.2948      0.1395
Q: Token 4      0.1928      0.4397      0.1781      0.1893

--- AFTER causal masking ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1      1.0000      0.0000      0.0000      0.0000
Q: Token 2      0.5558      0.4442      0.0000      0.0000
Q: Token 3      0.3602      0.2972      0.3426      0.0000
Q: Token 4      0.1928      0.4397      0.1781      0.1893

--- AFTER padding masking  (padding_mask = [1, 1, 1, 0]) ---
            K: Token 1  K: Token 2  K: Token 3  K: Token 4
Q: Token 1      0.3692      0.0804      0.5503         0.0
Q: Token 2      0.3424      0.2736      0.3841         0.0
Q: Token 3      0.3602      0.2972      0.3426         0.0
Q: Token 4      0.2379      0.5424      0.2197   

In [8]:
print("Context output with padding mask (rows = queries):")
print(pd.DataFrame(out_pad, index=[f"Q: {t}" for t in TOKENS]).to_string())
print()
print("Context output with causal + padding masks:")
print(pd.DataFrame(out_both, index=[f"Q: {t}" for t in TOKENS]).to_string())

Context output with padding mask (rows = queries):
                 0       1       2       3       4       5       6       7
Q: Token 1  0.4400 -0.0314 -0.3171 -0.3253 -0.3757  0.2101 -0.6790 -0.4364
Q: Token 2  0.1674  0.1534 -0.2199 -0.1160 -0.2824  0.1649 -0.4459 -0.4678
Q: Token 3  0.1394  0.2068 -0.2076 -0.0921 -0.2276  0.1450 -0.4344 -0.4959
Q: Token 4  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000

Context output with causal + padding masks:
                 0       1       2       3       4       5       6       7
Q: Token 1  0.7112  0.7933 -0.3487 -0.4624  0.8580 -0.1913 -1.2757 -1.1333
Q: Token 2 -0.0130  0.6618 -0.1306  0.0497  0.2871 -0.0359 -0.4312 -0.7673
Q: Token 3  0.1394  0.2068 -0.2076 -0.0921 -0.2276  0.1450 -0.4344 -0.4959
Q: Token 4  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000


In [9]:
# Automated checks
for name, w in [("padding", w_pad), ("causal+padding", w_both)]:
    assert np.allclose(w[:, 3], 0.0), f"{name}: padded key got attention"
    assert np.allclose(w.sum(axis=-1), 1.0), f"{name}: rows must sum to 1"
for name, out in [("padding", out_pad), ("causal+padding", out_both)]:
    assert np.allclose(out[3], 0.0), f"{name}: padded query output not zero"
    assert not np.allclose(out[:3], 0.0), f"{name}: valid outputs should be non-zero"
assert np.allclose(w_both[np.triu_indices(seq_len, k=1)], 0.0)
assert np.allclose(w_causal, weights_b)  # (c) reproduces (b)
assert np.allclose(w_none, weights_a)    # (c) reproduces (a)

print("Padded key column (Token 4) weights:", w_pad[:, 3])
print("Padded query row (Token 4) output:  ", out_pad[3])
print("All part (c) checks passed.")

Padded key column (Token 4) weights: [0. 0. 0. 0.]
Padded query row (Token 4) output:   [0. 0. 0. 0. 0. 0. 0. 0.]
All part (c) checks passed.


## Causal Masking vs. Padding Masking

**Causal masking** enforces *time order*. It is a fixed lower-triangular pattern that depends only on sequence length: token $i$ can attend to tokens $1 \dots i$ and never to later tokens. It is used in decoders and autoregressive language models so that, during training, a token's prediction cannot peek at the tokens it is supposed to predict. Every row keeps at least one allowed position (the diagonal), and the mask is the same for every sequence in a batch.

**Padding masking** enforces *validity*. It depends on the data: sequences in a batch are padded to the same length, and the mask (1 = real token, 0 = padding) differs per sequence. It blocks whole **columns** so that no query attends to a padding key, and the outputs of padded **query rows** are set to zero because they carry no meaning. It is used in both encoders and decoders whenever batches contain variable-length sequences.

In short, causal masking hides tokens that are real but *in the future*; padding masking hides tokens that are *not real at all*. When both are needed, they are combined with a logical AND, as in the `causal + padding` matrix above: the upper triangle is zero (causal) and the Token 4 column is zero (padding).

One implementation note: masking uses $-\infty$ added before softmax rather than zeroing weights after softmax. Zeroing afterward would leave rows that no longer sum to 1, whereas $-\infty$ makes softmax renormalize over the allowed positions only. The `softmax` helper also guards against rows where every key is masked, returning zeros instead of NaN.